# Numerical Comparison of Restart Methods of the Arnoldi Method

## Imports

In [ ]:
using LinearAlgebra
using JacobiDavidson
using LinearMaps
using MatrixDepot
ENV["GKSwstype"] = "nul"
using Plots
using Plots.Measures
using ProgressMeter
using LaTeXStrings

include("../src/Orthogonalization.jl")
include("../src/Arnoldi.jl")
include("../src/ImplicitRestart.jl")
include("../src/Eigenpairs.jl")
include("../src/BadRestart.jl")

## Experiments

We use the large sparse matrix "rajat12" from MatrixDepot to compare the accuracy of the Naive Restart and IRAM as a function of the number of iterations and subspace dimension.

In [ ]:
A = matrixdepot(r"rajat12") + 10 * I
n = size(A, 1)
num_eig = 4

display(A)

### Accuracy: Naive Restart vs IRAM

In [ ]:
maximum_iterations = 50
maximum_subspace = 30
mean_iterations = 5

iter_grid = Int.(range(1, maximum_iterations, maximum_iterations))
subspace_grid = Int.(range(6, maximum_subspace, 13))

residuals_iram = zeros(length(iter_grid), length(subspace_grid))
residuals_naive = zeros(length(iter_grid), length(subspace_grid))

it = 1

@showprogress for s in subspace_grid
    for _ in 1:mean_iterations
        _, _, residual_iram = Eigenpairs.eigenpairs_iram(A, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
        _, _, residual_naive = Eigenpairs.eigenpairs_naive_restart(A, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
        residuals_iram[:, it] += residual_iram
        residuals_naive[:, it] += residual_naive
    end
    it += 1
end

residuals_iram ./= mean_iterations
residuals_naive ./= mean_iterations;

In [ ]:
println("Range of IRAM residuals: ", extrema(log10.(residuals_iram)))
println("Range of Naive residuals: ", extrema(log10.(residuals_naive)))

In [ ]:
p = surface(iter_grid, subspace_grid, log10.(residuals_iram'), 
        xlabel="Restart iterations              ", 
        ylabel="Subspace dimensions", 
        zlabel="Residual",
        title="Log Residuals of IRAM",
        colorbar = true,
        alpha = 1,
        zlim=(-12, 2),
        camera=(60, 30),
        ratio=:equal,
        right_margin = 12 * Measures.mm)

savefig(p, "../fig/Naive vs IRAM/Residuals_IRAM.png")

In [ ]:
p = surface(iter_grid, subspace_grid, log10.(residuals_naive'), 
        xlabel="Restart iterations              ", 
        ylabel="Subspace dimensions", 
        zlabel="Residual",
        title="Log Residuals of Naive Restart",
        colorbar = true,
        alpha = 1,
        zlim=(-12, 2),
        camera=(60, 30),
        ratio=:equal,
        right_margin = 12 * Measures.mm)

savefig(p, "../fig/Naive vs IRAM/Residuals_Naive.png")

In [ ]:
p = plot(iter_grid, residuals_iram[:, end], color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Restart iterations", ylabel="Residual",
     yaxis=:log,
     xlim=[1, maximum_iterations], ylim = [1e-12, 1e6], legend=:top,
     label = "IRAM (dimension 50)")

plot!(p, iter_grid, residuals_naive[:, end], color=:green,
     label = "Naive (dimension 50)")

plot!(p, iter_grid, residuals_iram[:, 1], color=:red,
     label = "IRAM (dimension 6)")

plot!(p, iter_grid, residuals_naive[:, 1], color=:orange,
     label = "Naive (dimension 6)")

savefig(p, "../fig/Naive vs IRAM/Residuals_subspace.png")

In [ ]:
p = plot(subspace_grid, residuals_iram[end, :], color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Krylov subspace dimensions", ylabel="Residual",
     yaxis=:log,
     xlim=[6, maximum_subspace], ylim = [1e-12, 1e2], legend=:top,
     label = "IRAM")

plot!(p, subspace_grid, residuals_naive[end, :], color=:green,
     label = "Naive")

savefig(p, "../fig/Naive vs IRAM/Residuals_iteration.png")